## Instalo librerias necesarias!

In [ ]:
!pip install netCDF4
!pip install numpy
!pip install xarray
!pip install matplotlib

# instalar compiladores necesarios
!apt-get update
!apt-get install -y gfortran

# instalar wrf-python
#!pip install wrf-python ESTA LIBRERIA ES LO OPTIMO AUNQUE NO FUNCIONA EN COLAB !!! EN VS CODE DEBERIA ANDAR


## importo librerias

In [ ]:
from netCDF4 import Dataset
import glob
import numpy as np
import matplotlib.pyplot as plt
#from wrf import getvar, vertcross, interpline, CoordPair
import xarray as xr ## usaremos xr en vez de wrf


##

In [ ]:
wrf_list='/content/drive/MyDrive/AÑO 2026/TrabajoFinalCamilo/CompartidoCamilo/wrfout*'
wrf_list=glob.glob(wrf_list)
wrf_list.sort()

for file in wrf_list[0:1]:

     ds = xr.open_dataset(file)



# ver variables disponibles
print(list(ds.data_vars))


In [ ]:
wrf_list

In [ ]:
# variables asociadas al hielo

qhail = ds["QHAIL"]   # kg/kg
qgraup = ds["QGRAUP"] # kg/kg


# suma
qtotal = qhail + qgraup

## calculo de altura
g = 9.81

ph = ds["PH"]
phb = ds["PHB"]

z = (ph + phb) / g   # altura en metros

z_stag = (ds["PH"] + ds["PHB"]) / 9.81

# promedio entre niveles → ahora coincide con qtotal
z = 0.5 * (z_stag[:, :-1, :, :] + z_stag[:, 1:, :, :])

# elegí dos puntos en la grilla
y0, x0 = 100, 100
y1, x1 = 140, 140

npoints = 100

ys = np.linspace(y0, y1, npoints).astype(int)
xs = np.linspace(x0, x1, npoints).astype(int)

nz = qtotal.shape[1]  # niveles verticales

cross = np.zeros((nz, npoints))
height = np.zeros((nz, npoints))

for i in range(npoints):
    cross[:, i] = qtotal[0, :, ys[i], xs[i]]
    height[:, i] = z[0, :, ys[i], xs[i]]

fig, ax = plt.subplots(figsize=(10,6))

levels = np.linspace(0, np.max(cross), 20)

cf = ax.contourf(range(npoints), height[:,0], cross, levels=levels, cmap="turbo")

plt.colorbar(cf, ax=ax, label="QHAIL + QGRAUP (g/kg)")
ax.set_ylabel("Altura (m)")
ax.set_xlabel("Distancia (puntos)")

plt.title("Corte vertical QHAIL + QGRAUP")
plt.show()

In [ ]:
### variable electrica

light = ds["LIGHTDENS"] # kg/kg

# convertir a numpy 2D
light_2d = light[0, :, :].values
lats_2d = ds["XLAT"][0].values
lons_2d = ds["XLONG"][0].values

plt.figure(figsize=(8,6))

levels = np.linspace(0, 50, 15)

plt.contourf(lons_2d, lats_2d, light_2d, levels=levels, cmap="Blues",extend='max')
plt.colorbar(label="Lightning Density")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("LIGHTDENS")
plt.show()

In [ ]:
##### variable de reflectividad #####

reflectividad = ds["REFL_10CM"] # kg/kg

# convertir a numpy 2D
reflectividad_2d = np.max(reflectividad[0,:, :, :].values,axis=0)
lats_2d = ds["XLAT"][0].values
lons_2d = ds["XLONG"][0].values

plt.figure(figsize=(8,6))

levels = np.linspace(0, 50, 15)

plt.contourf(lons_2d, lats_2d, reflectividad_2d, levels=levels, cmap="Blues",extend='max')
plt.colorbar(label="reflectividad")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("Maxima reflectividad (COLMAX)")
plt.show()

In [ ]:
print(list(ds.data_vars))


In [ ]:
from netCDF4 import Dataset

# abrir archivo
ncfile = Dataset("/content/drive/MyDrive/wrfout_d01_2023-01-01_00:00:00")

# ver variables disponibles
print(ncfile.variables.keys())

# ejemplo: leer temperatura a 2m
t2 = ncfile.variables['T2'][:]

print(t2.shape)

